# Population-scale detection of putative embryonic mosaics from biobank CHIP calls

This notebook documents the analysis used to identify putative embryonic mosaic
individuals for canonical clonal hematopoiesis (CHIP) driver genes (e.g. *DNMT3A*,
*TET2*) from large biobank whole-genome sequencing data.

The logic is: individuals under 40 years old who carry a high–variant-allele-fraction
(VAF ≥ 0.25) somatic pathogenic variant in a CHIP driver gene are unlikely to have
reached that clone size through age-related clonal hematopoiesis, and are therefore
candidate embryonic mosaics.

**Inputs** are per-cohort CHIP call sets already generated with the Vlasschaert et al.
(*Blood* 2023) Mutect2 + ANNOVAR pipeline, with the binomial germline filter omitted so
that true near-heterozygous mosaic variants are not discarded.

No real participant data is included. Access to the real biobank data is governed by each program's data use
agreement.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

RNG = np.random.default_rng(0)
plt.style.use("default")

## 1. Input cohorts (AllofUs and BioVU)

## 2. Harmonize the two schemas

Each cohort is reduced to a common set of columns —
`age, gender, Gene.refGene, NonsynOI, AF, cohort` — so they can be concatenated.

Key cleaning steps:
- Drop comma-separated multi-allelic `AF` calls (ambiguous VAF).
- Explode the pipe-packed cohort so each clone becomes its own row.
- Coerce `AF`/`age` to numeric and drop rows missing either.

In [ ]:
COMMON_COLS = ["age", "gender", "Gene.refGene", "NonsynOI", "AF"]

def normalize_long(df):
    """Cohort A -> common schema."""
    out = df.copy()
    out = out[~out["AF"].astype(str).str.contains(",")]        # drop multi-allelic
    out = out[["Age", "Sex", "Gene.refGene", "NonsynOI", "AF"]].dropna()
    out.columns = COMMON_COLS
    out["gender"] = out["gender"].map({"Male": "M", "Female": "F"}).fillna(out["gender"])
    out["cohort"] = "cohort_a"
    return out

def normalize_packed(df):
    """Cohort B -> common schema (explode pipe-delimited clones)."""
    packed = ["chip_genes", "chip_variants", "chip_afs"]
    out = df.copy()
    for c in packed:
        out[c] = out[c].astype(str).str.split("|")
    out = out.explode(packed).drop_duplicates().reset_index(drop=True)
    out = out[["age_at_biosample", "gender", *packed]].dropna()
    out.columns = COMMON_COLS
    out["cohort"] = "cohort_b"
    return out

def harmonize(long_df, packed_df):
    combined = pd.concat([normalize_long(long_df), normalize_packed(packed_df)],
                         ignore_index=True)
    combined = combined.replace(["nan", "NaN", "None"], np.nan)
    combined["AF"] = pd.to_numeric(combined["AF"], errors="coerce")
    combined["age"] = pd.to_numeric(combined["age"], errors="coerce")
    combined = combined.dropna(subset=["AF", "age"]).drop_duplicates()
    return combined.reset_index(drop=True)

calls = harmonize(cohort_a, cohort_b)
print(f"{len(calls):,} harmonized variant calls across {calls['cohort'].nunique()} cohorts")
calls.head()

## 3. Per-gene mosaic classification

For a given driver gene:

1. Subset to that gene.
2. (Optional) apply gene-specific exclusions for known problematic loci — e.g. artifact-prone
   variants. In the real analysis a small number of specific hotspot/artifact calls were removed.
3. Classify **putative mosaics** as `age < 40` and `VAF ≥ 0.25`.

The age (40) and VAF (0.25) thresholds are the study definition and are set once here.

In [ ]:
AGE_CUTOFF = 40
VAF_CUTOFF = 0.25

def process_gene(calls, gene, exclude_variants=None):
    """Return (all calls for gene, putative mosaic subset)."""
    sub = calls[calls["Gene.refGene"] == gene].copy()
    if exclude_variants:
        sub = sub[~sub["NonsynOI"].isin(exclude_variants)]
    sub = sub.dropna(subset=["NonsynOI"])
    is_mosaic = (sub["age"] < AGE_CUTOFF) & (sub["AF"] >= VAF_CUTOFF)
    return sub, sub[is_mosaic].copy()

d3a_all, d3a_mosaics = process_gene(calls, "DNMT3A")
tet2_all, tet2_mosaics = process_gene(calls, "TET2")

print(f"DNMT3A: {len(d3a_mosaics)} putative mosaics of {len(d3a_all)} carriers")
print(f"TET2:   {len(tet2_mosaics)} putative mosaics of {len(tet2_all)} carriers")
d3a_mosaics.sort_values("age").head()

## 4. Age vs. VAF plot

Each carrier is a point; putative mosaics (young, high-VAF) are highlighted in red. A linear
trend line summarizes the expected positive age–VAF relationship of age-related CHIP.

In [ ]:
def plot_age_vaf(gene_calls, gene, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))
    x = gene_calls["age"].to_numpy()
    y = gene_calls["AF"].to_numpy()
    mosaic = (x < AGE_CUTOFF) & (y >= VAF_CUTOFF)

    ax.scatter(x[~mosaic], y[~mosaic], color="black", s=18, alpha=0.15)
    ax.scatter(x[mosaic],  y[mosaic],  color="red",   s=22, alpha=1.0,
               label="putative mosaic")

    if len(x) > 1:
        sl, ic, r, p, _ = stats.linregress(x, y)
        xs = np.array([x.min(), x.max()])
        ax.plot(xs, sl * xs + ic, color="blue", lw=1.5,
                label=f"R²={r**2:.4f}, p={p:.1e}")
        print(f"{gene}: R²={r**2:.4f}, p={p:.2e}")

    ax.axvline(AGE_CUTOFF, ls="--", color="grey", lw=0.8)
    ax.axhline(VAF_CUTOFF, ls="--", color="grey", lw=0.8)
    ax.set(xlim=(10, 90), ylim=(0, 1), xlabel="Age",
           ylabel="Variant allele fraction (VAF)",
           title=f"{gene} (n={len(x)})")
    ax.legend(loc="upper left", fontsize=8)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_age_vaf(d3a_all, "DNMT3A", axes[0])
plot_age_vaf(tet2_all, "TET2", axes[1])
plt.tight_layout()
plt.show()

## 5. Prevalence with Clopper–Pearson confidence intervals

Prevalence of putative mosaicism = (number of putative mosaics) / (number of individuals under
40 screened for that gene). The exact binomial **Clopper–Pearson** interval is used because the
event is rare, and results are reported per 10,000 people.

`n_screened` is the count of distinct individuals under 40 in the analyzed cohorts (the
denominator at risk), which in the real analysis was tracked from the demographic tables.

In [ ]:
def prevalence_per_10k(n_mosaics, n_screened, alpha=0.05):
    """Point estimate + Clopper-Pearson CI, expressed per 10,000 people."""
    lo, hi = proportion_confint(n_mosaics, n_screened, alpha=alpha, method="beta")
    scale = 10_000
    return {
        "n_mosaics": n_mosaics,
        "n_screened": n_screened,
        "per_10k": n_mosaics / n_screened * scale,
        "ci_low_per_10k": lo * scale,
        "ci_high_per_10k": hi * scale,
    }

# denominator: distinct individuals under 40 in the harmonized set
n_under_40 = calls.loc[calls["age"] < AGE_CUTOFF]
# (in the real analysis this comes from the demographic tables, one row per person)
n_screened = n_under_40[["cohort", "age", "gender"]].drop_duplicates().shape[0]

summary = pd.DataFrame([
    {"gene": "DNMT3A", **prevalence_per_10k(len(d3a_mosaics), n_screened)},
    {"gene": "TET2",   **prevalence_per_10k(len(tet2_mosaics), n_screened)},
])
summary.round(2)